# Train MultiVI from an ATAC-only Lamin collection

This tutorial shows how to create a Lamin collection containing ATAC-seq `AnnData` artifacts and train `MULTIVI` with `MultiVIMappedCollectionDataModule` when RNA data is unavailable.

## 1. Create a LaminDB instance and ATAC-only collection

Below we build a small ATAC-only example collection by writing `.h5ad` artifacts and registering them in LaminDB.

In [ ]:
from pathlib import Path

import lamindb as ln
from scvi.data import synthetic_iid

# Initialize a local LaminDB instance (run once per storage location).
ln.setup.init(storage="./lamin_atac_storage")

# Create synthetic multimodal data and keep only the ATAC modality.
mdata = synthetic_iid(
    n_batches=2,
    batch_size=64,
    n_genes=100,
    n_regions=200,
    return_mudata=True,
)
atac = mdata.mod["accessibility"].copy()
atac.obs["batch"] = atac.obs["batch"].astype(str)

# Split into multiple artifacts to mimic a collection-backed dataset.
artifact_dir = Path("./atac_artifacts")
artifact_dir.mkdir(parents=True, exist_ok=True)
artifacts = []
for idx, subset in enumerate([atac[:64].copy(), atac[64:].copy()]):
    path = artifact_dir / f"atac_part_{idx}.h5ad"
    subset.write_h5ad(path)
    artifacts.append(ln.Artifact(path).save())

atac_collection = ln.Collection(
    artifacts=artifacts,
    name="my_atac_collection",
).save()


## 2. Build a `MultiVIMappedCollectionDataModule` with ATAC only

Set `rna_collection=None` and pass only `atac_collection`.

In [ ]:
from scvi.dataloaders import MultiVIMappedCollectionDataModule

datamodule = MultiVIMappedCollectionDataModule(
    rna_collection=None,
    atac_collection=atac_collection,
    batch_key="batch",
    batch_size=128,
    shuffle=True,
)

datamodule.n_obs, datamodule.n_vars, datamodule.n_regions


## 3. Initialize and train `MULTIVI` from the datamodule registry

When using this custom dataloader path, initialize `MULTIVI` with `adata=None` and `registry=datamodule.registry`.

In [ ]:
from scvi.model import MULTIVI

model = MULTIVI(
    adata=None,
    registry=datamodule.registry,
    modality_weights="equal",
)

model.train(datamodule=datamodule, max_epochs=10)


## 4. Compute latent representations

Use the datamodule inference dataloader to obtain latent embeddings for all cells.

In [ ]:
latent = model.get_latent_representation(
    dataloader=datamodule.inference_dataloader(batch_size=128)
)
latent.shape
